# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution Exploration with `mlcroissant`

This notebook demonstrates loading and exploring the FAIR² dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL:

`https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json`

In [ ]:
# Ensure `mlcroissant` is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import pprint

# Define the dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the Croissant dataset
dataset = mlc.Dataset(croissant_url)

# Print some of the metadata
print(f"Name: {dataset.metadata.name}")
print(f"Description: {dataset.metadata.description}")
print(f"Citation: {dataset.metadata.cite_as}")


## 2. Data Overview
Review available record sets, fields, and their IDs.

We'll iterate through record sets, and for each record set, list its `@id` and the `@id`s of its fields.

In [ ]:
# List all record sets in the dataset with their @id's
print('Available record sets:')
record_sets = list(dataset.record_sets)
for rs in record_sets:
    print(f"  • Record Set @id: {rs.id} (name: {rs.name})")
    if hasattr(rs, 'fields'):
        for field in rs.fields:
            print(f"      - Field @id: {field.id} (name: {field.name}, dtype: {field.data_type})")
    else:
        print("    No fields found.")

## 3. Data Extraction
Load data from each record set into a pandas DataFrame for further exploration.

We'll use the record set and field `@id`s as shown above. 

For this dataset, most of the content should be in the *main clinical table* record set.

In [ ]:
# Choose record sets for extraction
# (Please review the output above for actual record set @id's)
record_set_ids = [rs.id for rs in dataset.record_sets]

# Load each record set into a DataFrame
dataframes = {}

for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    if records:
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"Loaded {len(df)} records from '{record_set_id}'")
    else:
        print(f"No records found for '{record_set_id}' (possibly a metadata or empty set).")

# Show columns of the main clinical table - pick the table with the most columns/rows
main_table_id = None
max_columns = 0
for rs_id, df in dataframes.items():
    if len(df.columns) > max_columns:
        main_table_id = rs_id
        max_columns = len(df.columns)

if main_table_id:
    print(f"\nMain Table record set @id: {main_table_id}")
    print("Columns:", dataframes[main_table_id].columns.tolist())
    display(dataframes[main_table_id].head())
else:
    print("No main table found with records.")

## 4. Exploratory Data Analysis (EDA)
Apply some common data cleaning/processing steps. We'll select key numeric and categorical fields (using their `@id` columns), filter, normalize, and group the data.

In [ ]:
# Use the main clinical table from above
df = dataframes.get(main_table_id)

# Identify a numeric field; use its @id as column name
# For illustration, we'll look for columns containing e.g. 'age' or 'interval' as common clinical numerics
numeric_field_candidates = [col for col in df.columns if any(x in col.lower() for x in ["age", "interval", "number", "years", "months"])]
print("Numeric field candidates:", numeric_field_candidates)

# Select first suitable as numeric_field_id
numeric_field_id = numeric_field_candidates[0] if numeric_field_candidates else df.columns[0]
print(f"Selected numeric field: '{numeric_field_id}'")

# Attempt to convert to numeric (in case of string columns)
df[numeric_field_id] = pd.to_numeric(df[numeric_field_id], errors='coerce')
threshold = df[numeric_field_id].quantile(0.25)  # use a quartile for demo
filtered_df = df[df[numeric_field_id] > threshold]

print(f"Filtered records with {numeric_field_id} > {threshold:.2f}:")
display(filtered_df.head())

# Normalize
filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
print(f"\nNormalized {numeric_field_id} for filtered records:")
display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

# Try grouping by a categorical field, e.g. "sex" or "msi" or "location" or the first non-numeric column
from pandas.api.types import is_numeric_dtype
non_numeric_field_candidates = [col for col in df.columns if not is_numeric_dtype(df[col])]
# Pick the first with < 20 unique values (likely a clinical categorical)
group_field = None
for col in non_numeric_field_candidates:
    nuniq = df[col].nunique()
    if nuniq > 1 and nuniq < 20:
        group_field = col
        break

if group_field:
    print(f"\nGrouping by categorical field: '{group_field}'")
    grouped_df = filtered_df.groupby(group_field)[numeric_field_id].mean().reset_index()
    print("Mean of", numeric_field_id, "per", group_field)
    display(grouped_df.head())
else:
    print("No appropriate categorical field for grouping found.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

We'll plot the distribution of the chosen numeric field and relationship versus the group field (if found).

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Plot histogram of the numeric field
plt.figure(figsize=(7, 4))
sns.histplot(df[numeric_field_id].dropna(), bins=10, kde=True)
plt.title(f"Distribution of {numeric_field_id}")
plt.xlabel(numeric_field_id)
plt.ylabel("Count")
plt.show()

# If we have a valid group_field, plot boxplot
if group_field:
    plt.figure(figsize=(8, 5))
    sns.boxplot(x=group_field, y=numeric_field_id, data=filtered_df)
    plt.title(f"{numeric_field_id} by {group_field}")
    plt.xlabel(group_field)
    plt.ylabel(numeric_field_id)
    plt.xticks(rotation=30)
    plt.show()

## 6. Conclusion

This notebook demonstrated how to:
- Load clinical research data from a Croissant-described FAIR² dataset using `mlcroissant`.
- Review record sets, fields, and work directly with `@id` references.
- Extract the main clinical table and perform basic numerical and categorical summary analyses.
- Visualize field distributions.

Explore further by examining other fields, testing additional hypotheses, or combining dataset record sets for deeper discovery.